In [1]:
%load_ext dotenv
%dotenv ../.secrets

In [5]:
from pathlib import Path
from sklearn.datasets import fetch_20newsgroups

Path("dataset").mkdir(exist_ok=True)

documents_raw = fetch_20newsgroups(
    categories=['rec.sport.hockey'],
    subset='test', 
    remove=('headers', 'footers', 'quotes'))

for i, text in enumerate(documents_raw.data):
    Path(f"dataset/doc{i}.txt").write_text(text, encoding='utf-8')

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os

Path("chroma_db").mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(path="chroma_db")

chroma_client.delete_collection(name="20newsgroups")

collection = chroma_client.create_collection(
    name = "20newsgroups",
    embedding_function = OpenAIEmbeddingFunction(
        api_key = "any value",
        model_name="text-embedding-3-small",
        api_base='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}))

ids = []
documents = []

for p in Path("dataset").glob("*.txt"):
    text = p.read_text(encoding="utf-8")
    if text.strip():
        ids.append(p.stem)
        documents.append(text)

collection.add(documents = documents,
               ids = ids)

In [20]:
def guardrails(message):
    m = message.lower()
    if "cat" in m or "dog" in m:
        return "I can’t talk about cats or dogs."
    if "horoscope" in m or "zodiac" in m:
        return "I can’t talk about horoscopes or zodiac signs."
    if "taylor swift" in m:
        return "I can’t talk about Taylor Swift."
    if "system prompt" in m:
        return "I can’t share the system prompt."
    return None

In [17]:
import requests

def weather(message):
    if "weather" in message.lower():
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": 43.65, "longitude": -79.38, "current": "temperature_2m"})
        data = response.json()
        temp = data["current"]["temperature_2m"]
        return f"The temperature in Toronto is {temp}°C."
    return None

In [25]:
def semantic_search(message):
    if "search" in message.lower():
        query = message.lower().replace("search", "").strip()
        if query:
            col = chroma_client.get_collection("20newsgroups")
            docs = col.query(query_texts=[query], n_results=1).get("documents", [[]])
            if docs:
                return docs[0]
    return None

In [19]:
def calculator(message):
    if "calculate" in message.lower():
        expression = message.lower().replace("calculate", "").strip()
        if expression:
            try:
                return f"The result is {eval(expression)}."
            except:
                return "The expression is invalid."
    return None

In [26]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
import gradio as gr

llm = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")})

def simple_chat(message, history):
    refusal = guardrails(message)
    if refusal:
        return refusal

    for tool in (weather, semantic_search, calculator):
        output = tool(message)
        if output:
            return output

    langchain_messages = []
    for msg in history:
        if msg["role"] == "user":
            langchain_messages.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            langchain_messages.append(AIMessage(content=msg["content"]))
    langchain_messages.append(HumanMessage(content=message))

    response = llm.invoke(langchain_messages)

    return response.content

gr.ChatInterface(
    fn=simple_chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
